# UTS 31250 Introduction to Data Analytics

## Assessment Task 2: Data Exploration and Preparation

**Student Number:** 25740123

## P0. Data Audit and Setup

This section checks the structure and quality of the raw dataset before the assessed analysis begins. It verifies the dataset dimensions, missing values and several key consistency checks without modifying the original data.

P0 is a preparation and quality assurance step. It is not one of the assessed A or B tasks.

Two rules apply throughout this section. First, the raw file `25740123.csv` is never edited. Second, blank cells in the raw file are treated as missing values and are never read as zero. Nothing is imputed and no rows are removed here.

### P0.1 Import the libraries

Only pandas and numpy are needed for the audit. pandas is used to read the comma separated file and to summarise it, and numpy supports the numeric work. The library versions are printed so that the run can be reproduced later.

In [1]:
import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 3.0.3
numpy version: 2.4.6


### P0.2 Load the raw data

The raw file is read into a DataFrame named `df_raw`. This object is treated as read only for the whole notebook. Where a check needs a transformed value, a separate copy or Series is created instead of changing `df_raw`.

By default pandas reads an empty field as `NaN`, which is the correct behaviour here because a blank cell means the value was not recorded.

In [2]:
RAW_CSV = "25740123.csv"

df_raw = pd.read_csv(RAW_CSV)

print("Dataset shape:", df_raw.shape)
print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

Dataset shape: (4349, 25)
Number of rows: 4349
Number of columns: 25


The full ordered list of attribute names is printed next, because later sections refer to these attributes by name and the order matters when the audit table is read.

In [3]:
print("Ordered column names:")
for position, column_name in enumerate(df_raw.columns, start=1):
    print(position, column_name)

Ordered column names:
1 O_CITY
2 O_STATE
3 D_CITY
4 D_STATE
5 SCHEDULED_DEPARTURE
6 DEPARTURE_TIME
7 DEPARTURE_DELAY
8 TAXI_OUT
9 WHEELS_OFF
10 SCHEDULED_TIME
11 ELAPSED_TIME
12 AIR_TIME
13 DISTANCE
14 WHEELS_ON
15 TAXI_IN
16 SCHEDULED_ARRIVAL
17 ARRIVAL_TIME
18 ARRIVAL_DELAY
19 DIVERTED
20 CANCELLED
21 AIR_SYSTEM_DELAY
22 SECURITY_DELAY
23 WEATHER_DELAY
24 DATE
25 AIRLINE_NAME


A short preview of the first rows confirms that the file was read with the correct header row and that the columns are aligned as expected.

In [4]:
df_raw.head()

,O_CITY,O_STATE,D_CITY,D_STATE,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,WEATHER_DELAY,DATE,AIRLINE_NAME
0,Chicago,IL,Omaha,NE,1655,1706.0,11.0,8.0,1714.0,80,72.0,60.0,423,1814.0,4.0,1815,1818.0,3.0,0,0,NaN,NaN,NaN,2015-9-6,Southwest Airlines Co.
1,Ontario,CA,Salt Lake City,UT,1630,1625.0,-5.0,14.0,1639.0,108,100.0,81.0,558,1900.0,5.0,1918,1905.0,-13.0,0,0,NaN,NaN,NaN,2015-2-23,Skywest Airlines Inc.
2,Charlotte Amalie,VI,Atlanta,GA,1510,1526.0,16.0,11.0,1537.0,250,247.0,230.0,1599,1827.0,6.0,1820,1833.0,13.0,0,0,NaN,NaN,NaN,2015-1-16,Delta Air Lines Inc.
3,Salt Lake City,UT,Oakland,CA,1505,1505.0,0.0,14.0,1519.0,119,102.0,82.0,588,1541.0,6.0,1604,1547.0,-17.0,0,0,NaN,NaN,NaN,2015-12-1,Delta Air Lines Inc.
4,Fargo,ND,Denver,CO,1712,1805.0,53.0,11.0,1816.0,123,125.0,95.0,627,1851.0,19.0,1815,1910.0,55.0,0,0,2.0,0.0,0.0,2015-2-4,Skywest Airlines Inc.


### P0.3 Attribute audit table

This table gives one row per attribute and records the pandas data type, how many values are present, how many are missing, the missing percentage and how many distinct values occur. The missing percentage is calculated from the actual number of rows in the file rather than from an assumed row count.

The table is a new DataFrame, so building it does not change `df_raw`.

In [5]:
total_rows = len(df_raw)
audit_rows = []

for column_name in df_raw.columns:
    column = df_raw[column_name]
    missing_count = int(column.isna().sum())
    audit_rows.append({
        "Attribute": column_name,
        "Pandas dtype": str(column.dtype),
        "Non-missing count": int(column.notna().sum()),
        "Missing count": missing_count,
        "Missing percentage": round(100 * missing_count / total_rows, 2),
        "Unique values": int(column.nunique(dropna=True)),
    })

audit_table = pd.DataFrame(audit_rows)
audit_table

,Attribute,Pandas dtype,Non-missing count,Missing count,Missing percentage,Unique values
0,O_CITY,str,4349,0,0.00,213
1,O_STATE,str,4349,0,0.00,52
2,D_CITY,str,4349,0,0.00,214
3,D_STATE,str,4349,0,0.00,51
4,SCHEDULED_DEPARTURE,int64,4349,0,0.00,806
5,DEPARTURE_TIME,float64,4288,61,1.40,1084
6,DEPARTURE_DELAY,float64,4288,61,1.40,204
7,TAXI_OUT,float64,4284,65,1.49,73
8,WHEELS_OFF,float64,4284,65,1.49,1082
9,SCHEDULED_TIME,int64,4349,0,0.00,350


The audit table is saved so that it can be referenced later without rerunning the whole notebook.

In [6]:
audit_table.to_csv("outputs/P0_data_audit.csv", index=False)
print("Saved outputs/P0_data_audit.csv with", len(audit_table), "attribute rows.")

Saved outputs/P0_data_audit.csv with 25 attribute rows.


### P0.4 Duplicate rows

An exact duplicate row would mean the same flight record appears twice. This check counts rows that are identical across all 25 attributes.

In [7]:
duplicate_row_count = int(df_raw.duplicated().sum())
print("Exact duplicate rows:", duplicate_row_count)

Exact duplicate rows: 0


### P0.5 Missing values and unique values for every attribute

The audit table above already contains this information, but printing the two series separately makes the pattern of missing data easier to read.

In [8]:
print("Missing values per attribute:")
print(df_raw.isna().sum())

Missing values per attribute:
O_CITY                    0
O_STATE                   0
D_CITY                    0
D_STATE                   0
SCHEDULED_DEPARTURE       0
DEPARTURE_TIME           61
DEPARTURE_DELAY          61
TAXI_OUT                 65
WHEELS_OFF               65
SCHEDULED_TIME            0
ELAPSED_TIME             84
AIR_TIME                 84
DISTANCE                  0
WHEELS_ON                67
TAXI_IN                  67
SCHEDULED_ARRIVAL         0
ARRIVAL_TIME             67
ARRIVAL_DELAY            84
DIVERTED                  0
CANCELLED                 0
AIR_SYSTEM_DELAY       3522
SECURITY_DELAY         3522
WEATHER_DELAY          3522
DATE                      0
AIRLINE_NAME              0
dtype: int64


In [9]:
print("Unique values per attribute:")
print(df_raw.nunique(dropna=True))

Unique values per attribute:
O_CITY                  213
O_STATE                  52
D_CITY                  214
D_STATE                  51
SCHEDULED_DEPARTURE     806
DEPARTURE_TIME         1084
DEPARTURE_DELAY         204
TAXI_OUT                 73
WHEELS_OFF             1082
SCHEDULED_TIME          350
ELAPSED_TIME            353
AIR_TIME                334
DISTANCE                930
WHEELS_ON              1126
TAXI_IN                  50
SCHEDULED_ARRIVAL      1024
ARRIVAL_TIME           1133
ARRIVAL_DELAY           236
DIVERTED                  2
CANCELLED                 2
AIR_SYSTEM_DELAY         81
SECURITY_DELAY            5
WEATHER_DELAY            44
DATE                    334
AIRLINE_NAME              6
dtype: int64


### P0.6 Cancelled and diverted flights

`CANCELLED` and `DIVERTED` are indicator attributes, so they should only contain the values 0 and 1. The number of flights flagged in each attribute is counted.

In [10]:
print("CANCELLED unique values:", sorted(df_raw["CANCELLED"].unique()))
print("DIVERTED unique values:", sorted(df_raw["DIVERTED"].unique()))

cancelled_count = int((df_raw["CANCELLED"] == 1).sum())
diverted_count = int((df_raw["DIVERTED"] == 1).sum())

print("Rows where CANCELLED == 1:", cancelled_count)
print("Rows where DIVERTED == 1:", diverted_count)

CANCELLED unique values: [np.int64(0), np.int64(1)]
DIVERTED unique values: [np.int64(0), np.int64(1)]
Rows where CANCELLED == 1: 66
Rows where DIVERTED == 1: 18


### P0.7 Airlines

The number of distinct airline names is counted and the names are listed, so that the categorical attribute can be described accurately in the later sections.

In [11]:
airline_names = sorted(df_raw["AIRLINE_NAME"].dropna().unique())
number_of_airlines = int(df_raw["AIRLINE_NAME"].nunique(dropna=True))

print("Number of airlines:", number_of_airlines)
for airline in airline_names:
    print("-", airline)

Number of airlines: 6
- American Airlines Inc.
- Atlantic Southeast Airlines
- Delta Air Lines Inc.
- Skywest Airlines Inc.
- Southwest Airlines Co.
- United Air Lines Inc.


### P0.8 Date range

`DATE` is stored as text in the raw file, and the values are not zero padded, for example `2015-9-6`. To check the date range the text is parsed into a temporary Series called `date_parsed` using the explicit format `%Y-%m-%d`. Giving the format explicitly avoids any guessing about whether the day or the month comes first.

`df_raw["DATE"]` itself is left as text, and the last two lines confirm that.

In [12]:
print("Original DATE dtype:", df_raw["DATE"].dtype)

date_parsed = pd.to_datetime(df_raw["DATE"], format="%Y-%m-%d")

print("Parsed minimum date:", date_parsed.min().date())
print("Parsed maximum date:", date_parsed.max().date())
print("Dates that failed to parse:", int(date_parsed.isna().sum()))

print("DATE dtype in df_raw after parsing:", df_raw["DATE"].dtype)
print("First DATE value in df_raw:", repr(df_raw["DATE"].iloc[0]))

Original DATE dtype: str
Parsed minimum date: 2015-01-01
Parsed maximum date: 2015-12-31
Dates that failed to parse: 0
DATE dtype in df_raw after parsing: str
First DATE value in df_raw: '2015-9-6'


### P0.9 ARRIVAL_DELAY audit

`ARRIVAL_DELAY` is the main outcome attribute for this dataset, measured in minutes, where a negative value means the flight arrived early. Missing values are counted as missing and are never treated as a delay of zero. All statistics below are therefore calculated on the non-missing values only, which is what pandas does by default.

In [13]:
arrival_delay = df_raw["ARRIVAL_DELAY"]

arrival_delay_non_missing = int(arrival_delay.notna().sum())
arrival_delay_min = float(arrival_delay.min())
arrival_delay_max = float(arrival_delay.max())
arrival_delay_negative = int((arrival_delay < 0).sum())

print("Non-missing ARRIVAL_DELAY values:", arrival_delay_non_missing)
print("Missing ARRIVAL_DELAY values:", int(arrival_delay.isna().sum()))
print("Minimum ARRIVAL_DELAY:", arrival_delay_min, "minutes")
print("Maximum ARRIVAL_DELAY:", arrival_delay_max, "minutes")
print("Non-missing values below zero (early arrivals):", arrival_delay_negative)

Non-missing ARRIVAL_DELAY values: 4265
Missing ARRIVAL_DELAY values: 84
Minimum ARRIVAL_DELAY: -61.0 minutes
Maximum ARRIVAL_DELAY: 865.0 minutes
Non-missing values below zero (early arrivals): 2633


### P0.10 Why the values are missing

The missing values in this dataset are not random. A cancelled flight never departs and a diverted flight never lands at the scheduled destination, so no arrival time can be recorded. This section tests whether the rows with a missing `ARRIVAL_DELAY` are exactly the rows that were cancelled or diverted.

Counting alone is not enough, because two different sets of rows can still have the same size. The comparison below uses `Series.equals`, which compares the True and False pattern row by row, and it is confirmed with a comparison of the row index sets.

In [14]:
is_cancelled = df_raw["CANCELLED"] == 1
is_diverted = df_raw["DIVERTED"] == 1
cancelled_or_diverted = is_cancelled | is_diverted

arrival_delay_missing = df_raw["ARRIVAL_DELAY"].isna()
elapsed_time_missing = df_raw["ELAPSED_TIME"].isna()

print("Cancelled flights:", int(is_cancelled.sum()))
print("Diverted flights:", int(is_diverted.sum()))
print("Cancelled or diverted flights:", int(cancelled_or_diverted.sum()))
print("Rows with missing ARRIVAL_DELAY:", int(arrival_delay_missing.sum()))
print("Rows with missing ELAPSED_TIME:", int(elapsed_time_missing.sum()))

Cancelled flights: 66
Diverted flights: 18
Cancelled or diverted flights: 84
Rows with missing ARRIVAL_DELAY: 84
Rows with missing ELAPSED_TIME: 84


In [15]:
arrival_rows_match = bool(arrival_delay_missing.equals(cancelled_or_diverted))
elapsed_rows_match = bool(elapsed_time_missing.equals(cancelled_or_diverted))

flagged_index = set(df_raw.index[cancelled_or_diverted])
arrival_index_match = set(df_raw.index[arrival_delay_missing]) == flagged_index
elapsed_index_match = set(df_raw.index[elapsed_time_missing]) == flagged_index

print("Missing ARRIVAL_DELAY rows equal cancelled or diverted rows:", arrival_rows_match)
print("Missing ELAPSED_TIME rows equal cancelled or diverted rows:", elapsed_rows_match)
print("Same result using row index sets, ARRIVAL_DELAY:", arrival_index_match)
print("Same result using row index sets, ELAPSED_TIME:", elapsed_index_match)

Missing ARRIVAL_DELAY rows equal cancelled or diverted rows: True
Missing ELAPSED_TIME rows equal cancelled or diverted rows: True
Same result using row index sets, ARRIVAL_DELAY: True
Same result using row index sets, ELAPSED_TIME: True


### P0.11 The three delay-cause attributes

`AIR_SYSTEM_DELAY`, `SECURITY_DELAY` and `WEATHER_DELAY` break a late arrival down into its causes. These attributes have a very high number of missing values, so it is important to understand why before any later section uses them.

Two things are tested. First, whether the three attributes are missing on exactly the same rows. Second, whether the rows that do have values are exactly the rows where `ARRIVAL_DELAY` is 15 minutes or more, which is the usual reporting threshold for a delayed flight.

In [16]:
air_system_missing = df_raw["AIR_SYSTEM_DELAY"].isna()
security_missing = df_raw["SECURITY_DELAY"].isna()
weather_missing = df_raw["WEATHER_DELAY"].isna()

same_as_security = bool(air_system_missing.equals(security_missing))
same_as_weather = bool(air_system_missing.equals(weather_missing))

print("AIR_SYSTEM_DELAY missing on the same rows as SECURITY_DELAY:", same_as_security)
print("AIR_SYSTEM_DELAY missing on the same rows as WEATHER_DELAY:", same_as_weather)
print("Rows where the delay causes are recorded:", int((~air_system_missing).sum()))

AIR_SYSTEM_DELAY missing on the same rows as SECURITY_DELAY: True
AIR_SYSTEM_DELAY missing on the same rows as WEATHER_DELAY: True
Rows where the delay causes are recorded: 827


In [17]:
delay_15_or_more = df_raw["ARRIVAL_DELAY"] >= 15
delay_causes_recorded = ~air_system_missing

delay_cause_rows_match = bool(delay_causes_recorded.equals(delay_15_or_more))

print("Rows where ARRIVAL_DELAY is 15 minutes or more:", int(delay_15_or_more.sum()))
print("Recorded delay-cause rows equal those rows:", delay_cause_rows_match)

Rows where ARRIVAL_DELAY is 15 minutes or more: 827
Recorded delay-cause rows equal those rows: True


### P0.12 Validation table

The assignment brief describes what this dataset should look like. Every expected fact is recomputed from the raw file and compared with the expected value. The expected values are only used as a target for comparison, and no value is ever written back into the data to force a match.

In [18]:
expected_missing_counts = {
    "DEPARTURE_TIME": 61,
    "DEPARTURE_DELAY": 61,
    "TAXI_OUT": 65,
    "WHEELS_OFF": 65,
    "WHEELS_ON": 67,
    "TAXI_IN": 67,
    "ARRIVAL_TIME": 67,
    "ELAPSED_TIME": 84,
    "AIR_TIME": 84,
    "ARRIVAL_DELAY": 84,
    "AIR_SYSTEM_DELAY": 3522,
    "SECURITY_DELAY": 3522,
    "WEATHER_DELAY": 3522,
}

validation_rows = []

def record_check(check_name, expected_value, observed_value):
    validation_rows.append({
        "Check": check_name,
        "Expected": str(expected_value),
        "Observed": str(observed_value),
        "Match": str(expected_value) == str(observed_value),
    })

In [19]:
record_check("Dataset rows", 4349, df_raw.shape[0])
record_check("Dataset columns", 25, df_raw.shape[1])
record_check("Date minimum", "2015-01-01", date_parsed.min().date())
record_check("Date maximum", "2015-12-31", date_parsed.max().date())
record_check("Number of airlines", 6, number_of_airlines)

for column_name, expected_count in expected_missing_counts.items():
    record_check("Missing values in " + column_name,
                 expected_count,
                 int(df_raw[column_name].isna().sum()))

other_columns = [c for c in df_raw.columns if c not in expected_missing_counts]
record_check("Attributes expected to have no missing values",
             0,
             int(df_raw[other_columns].isna().sum().sum()))

In [20]:
record_check("Cancelled flights", 66, cancelled_count)
record_check("Diverted flights", 18, diverted_count)
record_check("Cancelled or diverted rows", 84, int(cancelled_or_diverted.sum()))
record_check("Missing ARRIVAL_DELAY count", 84, int(arrival_delay_missing.sum()))
record_check("ARRIVAL_DELAY non-missing count", 4265, arrival_delay_non_missing)
record_check("ARRIVAL_DELAY minimum", -61.0, arrival_delay_min)
record_check("ARRIVAL_DELAY maximum", 865.0, arrival_delay_max)
record_check("Negative ARRIVAL_DELAY count", 2633, arrival_delay_negative)
record_check("Missing ARRIVAL_DELAY row set equals cancelled or diverted", True, arrival_rows_match)
record_check("Missing ELAPSED_TIME row set equals cancelled or diverted", True, elapsed_rows_match)
record_check("Delay causes missing on identical rows", True, same_as_security and same_as_weather)
record_check("Delay-cause rows equal ARRIVAL_DELAY of 15 or more", True, delay_cause_rows_match)
record_check("Rows with ARRIVAL_DELAY of 15 or more", 827, int(delay_15_or_more.sum()))
record_check("Exact duplicate rows", 0, duplicate_row_count)

validation_table = pd.DataFrame(validation_rows)
validation_table

,Check,Expected,Observed,Match
0,Dataset rows,4349,4349,True
1,Dataset columns,25,25,True
2,Date minimum,2015-01-01,2015-01-01,True
3,Date maximum,2015-12-31,2015-12-31,True
4,Number of airlines,6,6,True
5,Missing values in DEPARTURE_TIME,61,61,True
6,Missing values in DEPARTURE_DELAY,61,61,True
7,Missing values in TAXI_OUT,65,65,True
8,Missing values in WHEELS_OFF,65,65,True
9,Missing values in WHEELS_ON,67,67,True


The last step of the validation is a single summary line. If any check had failed, the failing rows would be printed here so that the problem could be reported before any further work was done.

In [21]:
failed_checks = validation_table[validation_table["Match"] == False]

print("Total checks:", len(validation_table))
print("Checks passed:", int((validation_table["Match"] == True).sum()))
print("Checks failed:", len(failed_checks))

if len(failed_checks) == 0:
    print("All expected dataset facts were confirmed from the raw file.")
else:
    print("The following checks did not match the expected values:")
    print(failed_checks)

Total checks: 33
Checks passed: 33
Checks failed: 0
All expected dataset facts were confirmed from the raw file.


### P0.13 Raw file integrity

The final check confirms that the raw file was not altered by this notebook. The SHA-256 hash of `25740123.csv` is recalculated and compared with the hash recorded when the audit was first written. The file size and the row count are also reported. A matching hash means the file is byte for byte identical, so no value was written back, no row was removed and no blank cell was replaced.

In [22]:
RECORDED_CSV_HASH = "676681651e11de53308116df0fc5b2ef72c5b967ec779d0159f747edd10499de"

with open(RAW_CSV, "rb") as raw_file:
    current_csv_hash = hashlib.sha256(raw_file.read()).hexdigest()

print("Recorded SHA-256:", RECORDED_CSV_HASH)
print("Current SHA-256: ", current_csv_hash)
print("Raw file unchanged:", current_csv_hash == RECORDED_CSV_HASH)
print("Rows still loaded from the raw file:", len(df_raw))

Recorded SHA-256: 676681651e11de53308116df0fc5b2ef72c5b967ec779d0159f747edd10499de
Current SHA-256:  676681651e11de53308116df0fc5b2ef72c5b967ec779d0159f747edd10499de
Raw file unchanged: True
Rows still loaded from the raw file: 4349


### P0.14 Audit conclusion

The audit confirmed the structure the assignment brief describes. The file holds 4,349 flight records and 25 attributes, covering every date from 1 January 2015 to 31 December 2015 for six airlines, with no exact duplicate rows.

The missing values fall into two clear groups. The first group is the small number of operational attributes, from `DEPARTURE_TIME` through to `ARRIVAL_DELAY`, where between 61 and 84 values are absent. The 84 rows with no `ARRIVAL_DELAY` or `ELAPSED_TIME` are exactly the 66 cancelled and 18 diverted flights, so these values are missing because the flight did not complete rather than because of a recording error. The second group is the three delay-cause attributes, each missing 3,522 values on identical rows. Their 827 recorded rows are exactly the flights that arrived 15 minutes late or later, so a blank there means the flight was not reportably late, which is very different from a zero.

That distinction matters for the assessed sections. Reading any of these blanks as zero would understate delays and distort the summary statistics, so the raw missing values are carried forward unchanged and how to handle them will be decided as part of the later tasks.

No data was modified in this section. No values were imputed, no rows were removed and no charts were produced, because those belong to the assessed tasks that follow.